Electrode map generator
this interface automatically reads electrode layouts from dxf files,
and maps channels to PCB channels.

return: 3 x n_channels matrix (x-location, y-loction, PCB channel)

In [1]:
import numpy as np, matplotlib.pyplot as plt, ezdxf, csv
from pathlib import Path
from ipyfilechooser import FileChooser
import ipywidgets as widgets
import utils

In [2]:
doc = ezdxf.readfile("../electrode_map_example/example1/50um_pitch_compatible_with_FPC.dxf")
#doc = ezdxf.readfile("../electrode_map_example/example2/Xiaomin_electrode_v2.dxf")
layers = sorted({e.dxf.layer for e in doc.modelspace()})
print(layers)

['L2D0_etching2', 'L3D0_etching3', 'electrode', 'etching', 'metalization', 'polyimide', 'polyimide_etch_borders']


In [7]:
# give the layer name of the electrode contours in the DXF file
# if you don't know, give None and instead the number of contours to extract (the largest ones will be used)
layer_name = "electrodes"
# if you don't know the layer name, give None and instead the number of contours to extract (the one closest to the origin will be used)
# ATTENtiON: we assume that the last contour is a hook loop, which we will remove from the electrode layout
num_contours = 64

In [4]:
dxf_chooser = FileChooser(filter_pattern="*.dxf")
probe_chooser = FileChooser(filter_pattern="*.json")
headstage_chooser = FileChooser(filter_pattern="*.json")

display(widgets.HTML(
    "<h3>1. Electrode geometry</h3>"
    "<p>Select the electrode layout DXF file.</p>"
))
display(dxf_chooser)

display(widgets.HTML(
    "<h3>2. Probe mapping</h3>"
    "<p>Select the JSON mapping from electrode channels to connector pads.</p>"
))
display(probe_chooser)

display(widgets.HTML(
    "<h3>3. Headstage mapping</h3>"
    "<p>Select the JSON mapping from connector pads to headstage channels.</p>"
))
display(headstage_chooser)

HTML(value='<h3>1. Electrode geometry</h3><p>Select the electrode layout DXF file.</p>')

FileChooser(path='/home/mathilda/work/ElectrodeToGeometry/electrode2geometry2', filename='', title='', show_hi…

HTML(value='<h3>2. Probe mapping</h3><p>Select the JSON mapping from electrode channels to connector pads.</p>…

FileChooser(path='/home/mathilda/work/ElectrodeToGeometry/electrode2geometry2', filename='', title='', show_hi…

HTML(value='<h3>3. Headstage mapping</h3><p>Select the JSON mapping from connector pads to headstage channels.…

FileChooser(path='/home/mathilda/work/ElectrodeToGeometry/electrode2geometry2', filename='', title='', show_hi…

In [5]:
probe_json = Path(probe_chooser.value)
headstage_json = Path(headstage_chooser.value)

routing = utils.get_electrode_to_headstage_mapping(
    str(probe_json),
    str(headstage_json),
)

print(routing)

    electrode_channel  headstage_channel
0                   1                 51
1                   2                 53
2                   3                 48
3                   4                 37
4                   5                 61
..                ...                ...
59                 60                 20
60                 61                 12
61                 62                 27
62                 63                 29
63                 64                 28

[64 rows x 2 columns]


In [ ]:
run_button = widgets.Button(
    description="Generate Channel Map",
    button_style="success",
)
output = widgets.Output()

state = {
    "channel_map": None,
    "dxf_file": None,
}


def on_run_clicked(button):
    with output:
        output.clear_output(wait=True)

        if not dxf_chooser.value:
            print("Please select a DXF file")
            return

        try:
            print("Loading electrode layout...")

            xy = utils.load_and_plot_electrodes(
                dxf_chooser.value,
                layer_name=layer_name,
                num_contours=num_contours,
            )

            print("\nCreating channel map...")

            channel_map = np.full((3, len(routing)), np.nan)

            for _, row in routing.iterrows():
                electrode_channel = int(row["electrode_channel"])
                headstage_channel = row["headstage_channel"]
                electrode_index = electrode_channel - 1

                channel_map[0, electrode_index] = xy[electrode_index, 0]
                channel_map[1, electrode_index] = xy[electrode_index, 1]

                if headstage_channel is not None:
                    channel_map[2, electrode_index] = float(headstage_channel)

            state["channel_map"] = channel_map
            state["dxf_file"] = dxf_chooser.value

            print("\nSuccess!")
            print(f"Shape: {channel_map.shape}")
            print(f"Sample column: {channel_map[:, 0]} {channel_map[:, 1]} {channel_map[:, 2]}")

        except Exception as error:
            print(f"Error: {error}")


run_button.on_click(on_run_clicked)
display(run_button, output)

Button(button_style='success', description='Generate Channel Map', style=ButtonStyle())

Output()

In [10]:
run_button.close()
output.clear_output()

In [9]:
export_button = widgets.Button(
    description="Export to CSV",
    button_style="info",
)
export_output = widgets.Output()


def on_export_clicked(button):
    with export_output:
        export_output.clear_output(wait=True)

        if state["channel_map"] is None:
            print("Generate a channel map first")
            return

        try:
            export_file = utils.export_channel_map_to_csv(
                state["channel_map"],
                state["dxf_file"],
            )
            print(f"Exported to: {export_file}")
        except Exception as error:
            print(f"Export failed: {error}")


export_button.on_click(on_export_clicked)
display(export_button, export_output)

Button(button_style='info', description='Export to CSV', style=ButtonStyle())

Output()